# 17 · Reranking (Cross-Encoder)

Stage-2 rescoring of a shortlist. Includes hybrid candidates + rerank.

**Analogy handbook:** [reranking](../retriever-analogy-handbook.html#reranking)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


### Learning: Reranking with Cross-Encoder

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Reranking is a post-retrieval step that reorders an initial set of candidate documents to improve their relevance to the query. Cross-encoders are powerful models that can evaluate the relevance of a query-document pair more accurately than traditional methods by considering b...

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

Vector search
     ↓
Top 20 candidates

This diagram illustrates the process of an initial vector search to get top candidates.

### Learning: cross_encoder = HuggingFaceCrossEncoder(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** We initialize a `HuggingFaceCrossEncoder` model, specifying its name and device (`cpu` for demonstration). This cross-encoder will be used to score the relevance of retrieved documents.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sunny\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13097.84it/s]


### Learning: cross_encoder_reranker = CrossEncoderReranker(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** We wrap the cross-encoder model in a `CrossEncoderReranker` and specify `top_n=5` to keep only the top 5 reranked documents.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

### Learning: reranking_retriever = ContextualCompressionRetriever(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** A `ContextualCompressionRetriever` is used to apply the `CrossEncoderReranker` to the results of a `base_retriever` (our `candidate_retriever`).

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

### Learning: reranking_query = (

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** Define a query to test the reranking process.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
reranking_query = (
    "How did Meta collect and use human preference data to train Llama 2-Chat?"
)

### Learning: initial_candidates = candidate_retriever.invoke(

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** First, retrieve a larger set of initial candidates using the `candidate_retriever` before reranking. This provides the pool of documents for the cross-encoder to reorder.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

### Learning: initial_candidates

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Display the initial candidates retrieved, noting their original order based on the vector similarity.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
initial_candidates

[Document(id='577892fb-3838-4c4d-9978-4488765bf837', metadata={'producer': 'pdfTeX-1.40.25', 'year': 2023, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creationdate': '2023-07-20T00:30:36+00:00', 'paper_page': 10, 'paper': 'Llama 2', 'title': '', 'page': 9, 'keywords': '', 'section': 'fine_tuning', 'chunk_id': 'llama2-page-10-chunk-38', 'author': '', 'document_type': 'research_paper', 'moddate': '2023-07-20T00:30:36+00:00', 'total_pages': 77, 'subject': '', 'creator': 'LaTeX with hyperref', 'start_index': 2384, 'page_label': '10', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'trapped': '/False', 'organization': 'Meta', 'access_level': 'public'}, page_content='can be found in Section 4.2.1.\nHuman annotations were collected in batches on a weekly basis. As we collected more preference data, our\nreward models improved, and we were abl

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the initial candidates retrieved before reranking. This provides a baseline to compare against the reranked results.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)


Before Reranking: Initial Vector Candidates

RANK: 1
Paper page: 10
Section: fine_tuning
Chunk ID: llama2-page-10-chunk-38
----------------------------------------------------------------------------------------------------
can be found in Section 4.2.1.
Human annotations were collected in batches on a weekly basis. As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distribution, i.e., from
hyper-specialization (Scialom et al., 2020b), it is important before a newLlama 2-Chat tuning iteration to
gather new preference data using the latest

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
-------------------------------------------------------------------------------------------------

### Learning: reranked_documents = reranking_retriever.invoke(reranking_query)

**What you'll learn:** Rescore a shortlist with a query–document joint model.

**What this cell does:** Invoke the `reranking_retriever` with the query. This will first fetch candidates using the base retriever and then apply the cross-encoder reranker to select the top `n` most relevant documents.

**Watch for:** Rerankers are stage-2 only — never over the full corpus.



In [ ]:
reranked_documents = reranking_retriever.invoke(reranking_query)

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the documents after reranking. Observe how their order might have changed compared to the initial candidates, reflecting the cross-encoder's relevance assessment.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)


After Reranking: Final Top Documents

RANK: 1
Paper page: 10
Section: fine_tuning
Chunk ID: llama2-page-10-chunk-38
----------------------------------------------------------------------------------------------------
can be found in Section 4.2.1.
Human annotations were collected in batches on a weekly basis. As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distribution, i.e., from
hyper-specialization (Scialom et al., 2020b), it is important before a newLlama 2-Chat tuning iteration to
gather new preference data using the latest

RANK: 2
Paper page: 11
Section: fine_tuning
Chunk ID: llama2-page-11-chunk-45
----------------------------------------------------------------------------------------------------
t

User query
     ↓
Dense Retriever
     ↓
Top 20 candidates
     ↓
Cross-Encoder Reranker
     ↓
Query-document relevance evaluation
     ↓
Final top 5 documents

This diagram illustrates the complete reranking workflow: initial vector search, candidate selection, cross-encoder evaluation, and final top documents.

### Learning: Hybrid Retrieval + Reranking

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** This combines the benefits of hybrid retrieval (sparse + dense) for broad initial candidate generation with the precision of cross-encoder reranking. This is often the most robust retrieval strategy for complex RAG systems.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
bm25_retriever.k = 15

### Learning: dense_candidate_retriever = vector_store.as_retriever(

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Adjust `k` for the BM25 retriever to fetch more candidates, suitable for an initial broad retrieval.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

### Learning: hybrid_candidate_retriever = EnsembleRetriever(

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Configure a `dense_candidate_retriever` to fetch more candidates, complementing the BM25 retriever.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

### Learning: hybrid_reranking_retriever = ContextualCompressionRetriever(

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Create a `hybrid_candidate_retriever` by combining BM25 and dense retrievers using `EnsembleRetriever` with equal weights. This generates a diverse set of initial candidates.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

### Learning: query = (

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Wrap the `hybrid_candidate_retriever` with a `ContextualCompressionRetriever` and our `cross_encoder_reranker`. This pipeline will first retrieve candidates using hybrid search and then rerank them.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
query = (
    "What techniques did Meta use to improve the helpfulness and safety of Llama 2-Chat?"
)

### Learning: final_documents = hybrid_reranking_retriever.invoke(query)

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Define the final complex query to be used with the hybrid retrieval and reranking pipeline.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
final_documents = hybrid_reranking_retriever.invoke(query)

### Learning: final_documents

**What you'll learn:** Combine sparse + dense so exact and semantic hits both survive.

**What this cell does:** Invoke the `hybrid_reranking_retriever` with the query to get the final, highly relevant documents after both hybrid retrieval and cross-encoder reranking.

**Watch for:** Ask: how are the two lists fused — RRF, weights, or naive concat?



In [ ]:
final_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public', 'start_index': 823, 'chunk_id': 'llama2-page-1-chunk-1'}, page_content='Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang\nAngela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic\nSergey Edunov Thomas Scialom∗\nGenAI, Meta\nAbstract\nIn t

### Learning: display_documents(

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Display the `final_documents` to observe the results of the complete hybrid retrieval and reranking process.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)


Hybrid Retrieval + Cross-Encoder Reranking

RANK: 1
Paper page: 1
Section: front_matter
Chunk ID: llama2-page-1-chunk-1
----------------------------------------------------------------------------------------------------
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic
Sergey Edunov Thomas Scialom∗
GenAI, Meta
Abstract
In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned
large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters.
Our fine-tuned LLMs, calledLlama 2-Chat, are optimized for dialogue use cases. Our
models outperform open-source chat models on most benchmarks we tested, and based on
our human evaluations for helpfulness and 

RANK: 2
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-9
----------------------------------------------------------------------------------------------------

                         ┌── BM25 Search ───────┐
User query ──────────────┤                      ├── Weighted RRF
                         └── Dense Search ──────┘
                                                   ↓
                                          Candidate documents
                                                   ↓
                                       Cross-Encoder Reranker
                                                   ↓
                                           Final top documents

This diagram illustrates the comprehensive retrieval and reranking pipeline: initial hybrid search, candidate selection, cross-encoder reranking, and final top documents.